In [1]:
# =========================
# BLOCK 1 — PREPROCESSING (identical to Protein_ratios_noRFE.ipynb)
# =========================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# ---- File paths ----
BASE = "/Users/adithyamadduri/Downloads/syn65414912"
ANML_PATH   = os.path.join(BASE, "OhNM2025_ROSMAP_plasma_Soma7k_protein_level_ANML_log10.csv")
SAMPLE_PATH = os.path.join(BASE, "OhNM2025_ROSMAP_plasma_Soma7k_sample_metadata.csv")

# ---- Load ----
df_levels = pd.read_csv(ANML_PATH)        # rows: projid_visit, cols: proteins
df_meta   = pd.read_csv(SAMPLE_PATH)      # contains projid_visit, projid, msex, age_at_visit, educ, apoe_genotype, Diagnosis

# ---- Sanity: keys present? ----
assert "projid_visit" in df_levels.columns, "projid_visit missing in protein matrix."
for col in ["projid_visit","projid","msex","age_at_visit","educ","apoe_genotype","Diagnosis"]:
    assert col in df_meta.columns, f"{col} missing in sample metadata."

# ---- Align on visit ----
# inner join so we only keep visits present in BOTH
df = pd.merge(df_meta, df_levels, on="projid_visit", how="inner", validate="one_to_one")
print("Aligned shape:", df.shape)

# ---- Labels (four groups) ----
df["Diagnosis"] = df["Diagnosis"].astype(str).str.strip()
valid_classes = {"MCI","NCI","AD","AD+"}
df = df[df["Diagnosis"].isin(valid_classes)].reset_index(drop=True)
print("Class counts:\n", df["Diagnosis"].value_counts())

# ---- Grouping key for leakage control ----
df["projid"] = df["projid"].astype(str)

# ---- Stratification label: Diagnosis × sex (0/1) ----
df["msex"] = df["msex"].astype(int)
df["strata"] = df["Diagnosis"] + "_" + df["msex"].astype(str)

# ---- APOE one-hot (Unknown for NaN) ----
def format_apoe(x):
    if pd.isna(x):
        return "Unknown"
    try:
        # e.g., 33.0 -> "33"
        return str(int(float(x)))
    except Exception:
        s = str(x).strip()
        return s if s else "Unknown"

df["apoe_str"] = df["apoe_genotype"].apply(format_apoe)

# scikit-learn compatibility across versions
try:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")

apoe_ohe = ohe.fit_transform(df[["apoe_str"]])
apoe_cols = [c.replace("apoe_str_","APOE_") for c in ohe.get_feature_names_out()]
df_apoe  = pd.DataFrame(apoe_ohe, columns=apoe_cols, index=df.index)

# ---- Numeric covariates (unscaled) ----
for col in ["age_at_visit","educ"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---- Protein feature columns ----
protein_cols = [c for c in df_levels.columns if c != "projid_visit"]

# ---- Final feature matrix X ----
# NOTE: Demographics included here for consistency with the full preprocessing.
# The Z-scaled features fed to FLAML below are built from protein_cols only (no demographics),
# matching the ratio models which also exclude demographics.
X = pd.concat([df[["age_at_visit","educ"]], df_apoe, df[protein_cols]], axis=1)
y = df["Diagnosis"].astype(str).values
groups = df["projid"].values
strata = df["strata"].values

print("X shape:", X.shape)
print("APOE levels seen:", sorted(set(df["apoe_str"])))

Aligned shape: (973, 7301)
Class counts:
 Diagnosis
NCI    507
MCI    262
AD     167
AD+     17
Name: count, dtype: int64
X shape: (953, 7298)
APOE levels seen: ['22', '23', '24', '33', '34', '44', 'Unknown']


In [2]:
# =========================
# BLOCK 2 — Z-SCALE BASELINE TRAINING
# Uses the SAME protein shortlist from the SAME pre-trained LGBM pickle files,
# the SAME group-aware stratified splits, the SAME FLAML settings (lgbm, 1200s budget),
# and the SAME seeds as Protein_ratios_noRFE.ipynb.
#
# ONLY DIFFERENCE: Instead of generating pairwise log-ratios from the shortlisted proteins,
# we apply Z-scaling (standardization) to those same shortlisted proteins.
# Z-scale(protein_i) = (protein_i - column_mean) / column_std
# This removes mean differences and normalizes variance across proteins,
# providing a simpler multiplicative-bias-elimination strategy than log-ratios.
# =========================
import os
import pickle
import pandas as pd
from collections import Counter, defaultdict
import numpy as np
from flaml import AutoML
from sklearn.metrics import roc_auc_score

# ---- Load feature importances from the SAME pre-trained LGBM models ----
# This is identical to Protein_ratios_noRFE.ipynb
lgbm_model_dir = "/Users/adithyamadduri/Desktop/Projects/proteomics_LGBM(ANML+Meta)_fixed"
classes = ["MCI", "NCI", "AD", "AD+"]
seeds = [1, 2, 3, 4, 5]

def safe_cls(c): return c.replace("+", "plus").replace(" ", "_").replace("/", "-")

feature_counts_per_class = defaultdict(Counter)

for seed in seeds:
    for cls in classes:
        model_path = os.path.join(lgbm_model_dir, f"seed{seed}_{safe_cls(cls)}_automl.pkl")
        if not os.path.exists(model_path):
            print(f"Missing: {model_path}")
            continue

        with open(model_path, "rb") as f:
            automl = pickle.load(f)

        importances = automl.model.estimator.feature_importances_
        features = automl.feature_names_in_
        nonzero_features = [feat for feat, imp in zip(features, importances) if imp > 0]
        feature_counts_per_class[cls].update(nonzero_features)

# ---- Get shared features per class (appearing in >=2 models) ----
# Identical threshold to Protein_ratios_noRFE.ipynb
shared_features_per_class = {
    cls: [feat for feat, count in counter.items() if count >= 2]
    for cls, counter in feature_counts_per_class.items()
}

# ---- Output folder ----
out_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Zscale_baseline_LGBM"
os.makedirs(out_dir, exist_ok=True)

for cls in classes:
    feats = shared_features_per_class.get(cls, [])
    print(f"[{cls}] Found {len(feats)} features with non-zero importance in >=2 models.")

# ---- Custom group-aware stratified 70/30 split (identical to ratio pipeline) ----
def group_stratified_shuffle_split(df_index, strata_all, groups_all, test_size=0.30, random_state=0):
    rng = np.random.RandomState(random_state)
    data = pd.DataFrame({"idx": df_index, "strata": strata_all, "group": groups_all})
    grp_mode = data.groupby("group")["strata"].agg(lambda s: s.value_counts().idxmax())
    grp_mode = grp_mode.sample(frac=1.0, random_state=random_state)

    train_groups, test_groups = [], []
    for s_val, grp_ids in grp_mode.groupby(grp_mode.values):
        g_list = list(grp_ids.index)
        rng.shuffle(g_list)
        n_test = max(1, int(round(test_size * len(g_list))))
        test_groups.extend(g_list[:n_test])
        train_groups.extend(g_list[n_test:])

    train_mask = np.isin(groups_all, train_groups)
    test_mask = np.isin(groups_all, test_groups)
    return np.where(train_mask)[0], np.where(test_mask)[0]

# ---- FLAML training loop (per-class Z-scaled features) ----
for cls in classes:
    # 1) Use only predictive features (>=2 seeds) for THIS class — identical to ratio pipeline
    class_feats = shared_features_per_class.get(cls, [])
    protein_only_features = [f for f in class_feats if f in protein_cols]
    print(f"\n[{cls}] Using {len(protein_only_features)} protein features for Z-scale transformation...")

    if len(protein_only_features) < 2:
        print(f"[{cls}] Not enough protein features (need >=2). Skipping.")
        continue

    # 2) Z-scale transformation on the shortlisted proteins
    # DIFFERENCE FROM RATIO PIPELINE: Instead of pairwise log-ratios, we Z-scale each column.
    # Z-scale(protein_i) = (protein_i - column_mean) / column_std
    # This standardizes each protein across samples to zero mean, unit variance.
    X_shortlist = X[protein_only_features].copy()
    col_means = X_shortlist.mean(axis=0)
    col_stds = X_shortlist.std(axis=0)
    # Guard against zero-std columns (constant features); replace 0 std with 1 to avoid div-by-zero
    col_stds = col_stds.replace(0, 1)
    X_zscale_cls = (X_shortlist - col_means) / col_stds
    print(f"[{cls}] X_zscale shape: {X_zscale_cls.shape}")

    # 3) Seeded train/test + AutoML (one-vs-all for THIS class)
    # Identical splits, FLAML settings, time budget to ratio pipeline
    for seed in seeds:
        tr_idx, te_idx = group_stratified_shuffle_split(
            df_index=np.arange(len(X_zscale_cls)),
            strata_all=strata,
            groups_all=groups,
            test_size=0.30,
            random_state=seed,
        )

        X_train = X_zscale_cls.iloc[tr_idx]
        X_test = X_zscale_cls.iloc[te_idx]
        y_train_full = y[tr_idx]
        y_test_full = y[te_idx]

        print(f"[{cls}] [Seed {seed}] Train n={len(tr_idx)} | Test n={len(te_idx)}")

        y_train = (y_train_full == cls).astype(int)
        y_test = (y_test_full == cls).astype(int)

        automl = AutoML()
        settings = {
            "time_budget": 1200,
            "metric": "roc_auc",
            "task": "classification",
            "eval_method": "cv",
            "estimator_list": ["lgbm"],
            "log_file_name": os.path.join(out_dir, f"flaml_seed{seed}_{safe_cls(cls)}.log"),
            "seed": seed,
        }

        automl.fit(X_train=X_train, y_train=y_train, **settings)
        automl.pickle(os.path.join(out_dir, f"seed{seed}_{safe_cls(cls)}_automl.pkl"))

        y_score = automl.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_score) if (y_test.sum() > 0 and y_test.sum() < len(y_test)) else float("nan")
        print(f"  [{cls}] AUC={auc:.3f}")

        out_csv = os.path.join(out_dir, f"seed{seed}_{safe_cls(cls)}.csv")
        pd.DataFrame({
            "y_true": y_test.astype(int),
            "y_score": y_score.astype(float)
        }).to_csv(out_csv, index=False)

/var/folders/yr/4bplyc1x1tq4xz4jckbg_8wr0000gn/T/ipykernel_79580/3033164999.py:39: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  automl = pickle.load(f)
/var/folders/yr/4bplyc1x1tq4xz4jckbg_8wr0000gn/T/ipykernel_79580/3033164999.py:39: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usag

[MCI] Found 43 features with non-zero importance in >=2 models.
[NCI] Found 351 features with non-zero importance in >=2 models.
[AD] Found 156 features with non-zero importance in >=2 models.
[AD+] Found 23 features with non-zero importance in >=2 models.

[MCI] Using 43 protein features for Z-scale transformation...
[MCI] X_zscale shape: (953, 43)
[MCI] [Seed 1] Train n=670 | Test n=283
[flaml.automl.logger: 05-15 11:12:07] {1752} INFO - task = classification
[flaml.automl.logger: 05-15 11:12:07] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 05-15 11:12:07] {1862} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 05-15 11:12:07] {1979} INFO - List of ML learners in AutoML Run: ['lgbm']
[flaml.automl.logger: 05-15 11:12:07] {2282} INFO - iteration 0, current learner lgbm


/var/folders/yr/4bplyc1x1tq4xz4jckbg_8wr0000gn/T/ipykernel_79580/3033164999.py:39: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  automl = pickle.load(f)
/var/folders/yr/4bplyc1x1tq4xz4jckbg_8wr0000gn/T/ipykernel_79580/3033164999.py:39: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usag

[flaml.automl.logger: 05-15 11:12:07] {2417} INFO - Estimated sufficient time budget=496s. Estimated necessary time budget=0s.
[flaml.automl.logger: 05-15 11:12:07] {2466} INFO -  at 0.1s,	estimator lgbm's best error=0.3652,	best estimator lgbm's best error=0.3652
[flaml.automl.logger: 05-15 11:12:07] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 05-15 11:12:08] {2466} INFO -  at 0.1s,	estimator lgbm's best error=0.3214,	best estimator lgbm's best error=0.3214
[flaml.automl.logger: 05-15 11:12:08] {2282} INFO - iteration 2, current learner lgbm
[flaml.automl.logger: 05-15 11:12:08] {2466} INFO -  at 0.2s,	estimator lgbm's best error=0.3214,	best estimator lgbm's best error=0.3214
[flaml.automl.logger: 05-15 11:12:08] {2282} INFO - iteration 3, current learner lgbm
[flaml.automl.logger: 05-15 11:12:08] {2466} INFO -  at 0.2s,	estimator lgbm's best error=0.3214,	best estimator lgbm's best error=0.3214
[flaml.automl.logger: 05-15 11:12:08] {2282} INFO - iteration 4

In [3]:
# =========================
# BLOCK 3 — COMPARISON: Z-scale baseline vs PCRR ratios (noRFE)
# =========================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# -------------------------
# CONFIG
# -------------------------
zscale_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Zscale_baseline_LGBM"
ratio_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Protein_ratios_noRFE"

classes = ["MCI", "NCI", "AD", "AD+"]
seeds = [1, 2, 3, 4, 5]

def safe_cls(c):
    return c.replace("+", "plus").replace(" ", "_").replace("/", "-")

def compute_auc_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    y_true = df["y_true"].values
    y_score = df["y_score"].values

    # avoid crash if only one class appears in test split
    if np.sum(y_true) == 0 or np.sum(y_true) == len(y_true):
        return np.nan

    return roc_auc_score(y_true, y_score)

# -------------------------
# MAIN: compute mean AUC per class
# -------------------------
rows = []

for cls in classes:
    zscale_aucs = []
    ratio_aucs = []

    for seed in seeds:
        zscale_csv = os.path.join(zscale_dir, f"seed{seed}_{safe_cls(cls)}.csv")
        ratio_csv = os.path.join(ratio_dir, f"seed{seed}_{safe_cls(cls)}.csv")

        if os.path.exists(zscale_csv):
            zscale_aucs.append(compute_auc_from_csv(zscale_csv))
        else:
            zscale_aucs.append(np.nan)

        if os.path.exists(ratio_csv):
            ratio_aucs.append(compute_auc_from_csv(ratio_csv))
        else:
            ratio_aucs.append(np.nan)

    zscale_mean = np.nanmean(zscale_aucs)
    ratio_mean = np.nanmean(ratio_aucs)
    diff = ratio_mean - zscale_mean

    rows.append({
        "class": cls,
        "Zscale_LGBM": zscale_mean,
        "PCRR_ratios_LGBM": ratio_mean,
        "difference (PCRR - Zscale)": diff,
        "n_zscale": np.sum(~np.isnan(zscale_aucs)),
        "n_ratios": np.sum(~np.isnan(ratio_aucs)),
    })

# -------------------------
# OUTPUT TABLE
# -------------------------
out_df = pd.DataFrame(rows).set_index("class")

# keep only the 3 key columns
final_df = out_df[["Zscale_LGBM", "PCRR_ratios_LGBM", "difference (PCRR - Zscale)"]].copy()

print("\n=== Mean ROC AUC (5 seeds): Z-scale vs PCRR ===")
print(final_df.round(4))

# optional: show if any seeds were missing
if (out_df["n_zscale"] < len(seeds)).any() or (out_df["n_ratios"] < len(seeds)).any():
    print("\nNOTE: Some CSVs were missing (or had invalid AUC due to single-class test split).")
    print(out_df[["n_zscale", "n_ratios"]])


=== Mean ROC AUC (5 seeds): Z-scale vs PCRR ===
       Zscale_LGBM  PCRR_ratios_LGBM  difference (PCRR - Zscale)
class                                                           
MCI         0.6886            0.6457                     -0.0429
NCI         0.7594            0.7469                     -0.0125
AD          0.8473            0.8407                     -0.0066
AD+         0.8268            0.8136                     -0.0132
